In [1]:
%%html
<script>
(function() {
    var BG = '#e8f0fa';
    document.documentElement.style.setProperty('background', BG, 'important');
    document.body.style.setProperty('background', BG, 'important');
    document.body.style.setProperty('background-color', BG, 'important');

    var css = `
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

    html, body { background: #e8f0fa !important; font-family: 'Inter', sans-serif !important; }
    #notebook-container, .jp-Notebook, .jp-Cell,
    .jp-OutputArea, .jp-OutputArea-output, .jp-Cell-outputWrapper,
    .widget-output, .voila-container, .jp-RenderedHTMLCommon { background: transparent !important; }

    .game-hero {
        background: linear-gradient(135deg, #991b1b 0%, #b91c1c 45%, #7f1d1d 100%);
        border-radius: 14px; padding: 2rem 2.2rem; margin-bottom: 1rem;
        position: relative; overflow: hidden; box-shadow: 0 4px 24px rgba(153,27,27,0.25);
    }
    .game-hero::before {
        content: ''; position: absolute; top: 0; right: 0; bottom: 0; width: 40%;
        background: radial-gradient(ellipse at right center, rgba(255,255,255,0.08) 0%, transparent 65%);
        pointer-events: none;
    }
    .game-badge {
        display: inline-block; background: rgba(255,255,255,0.15);
        border: 1px solid rgba(255,255,255,0.3); border-radius: 20px;
        padding: 3px 14px; font-size: 0.72rem; font-weight: 600;
        color: #fff; margin-bottom: 0.9rem; letter-spacing: 0.8px; text-transform: uppercase;
    }
    .game-hero h1 {
        font-size: 1.8rem; font-weight: 800; margin: 0 0 0.5rem 0;
        letter-spacing: -0.5px; color: #ffffff;
    }
    .game-hero .accent { color: #fecaca; }
    .game-hero p { font-size: 0.86rem; color: rgba(255,255,255,0.78); margin: 0; font-weight: 300; line-height: 1.6; max-width: 720px; }
    .game-pill { background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.25); border-radius: 20px; padding: 4px 12px; font-size: 0.72rem; color: rgba(255,255,255,0.85); font-weight: 500; }
    .pill-row { display: flex; gap: 8px; margin-top: 1rem; flex-wrap: wrap; }

    .kpi-row { display: flex; gap: 10px; margin: 10px 0; flex-wrap: wrap; }
    .kpi-bio { flex: 1; min-width: 120px; background: #fff; border: 1px solid #c5d6ec; border-radius: 10px; padding: 0.9rem 1.1rem; box-shadow: 0 1px 4px rgba(37,99,235,0.07); }
    .kpi-bio-label { font-size: 0.64rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.8px; color: #93afc8; margin-bottom: 0.3rem; }
    .kpi-bio-val { font-size: 1.35rem; font-weight: 700; line-height: 1.1; margin-bottom: 0.2rem; }
    .kpi-bio-sub { font-size: 0.7rem; color: #93afc8; }

    .disease-card {
        background: #fff; border: 1px solid #c5d6ec; border-radius: 10px;
        padding: 1.1rem 1.3rem; box-shadow: 0 1px 4px rgba(37,99,235,0.07); margin: 8px 0;
    }
    .disease-name { font-size: 1.05rem; font-weight: 700; color: #1e3a5f; margin-bottom: 4px; }
    .disease-meta { font-size: 0.74rem; color: #6b8aaa; margin-bottom: 8px; font-style: italic; }
    .disease-stats { display: flex; gap: 12px; flex-wrap: wrap; margin: 8px 0; }
    .stat-chip { background: #f1f5f9; border: 1px solid #c5d6ec; border-radius: 6px; padding: 4px 10px; font-size: 0.74rem; color: #1e3a5f; }
    .stat-chip strong { color: #991b1b; }
    .disease-desc { font-size: 0.82rem; color: #4a6a8a; line-height: 1.55; margin-top: 6px; }
    `;

    var style = document.createElement('style');
    style.textContent = css;
    document.head.appendChild(style);

    var n = 0;
    var iv = setInterval(function() {
        document.body.style.setProperty('background','#e8f0fa','important');
        if (++n >= 10) clearInterval(iv);
    }, 300);
})();
</script>


In [2]:
import random, math, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore')

BG='#e8f0fa'; CARD='#ffffff'; BORD='#c5d6ec'
P1='#2563eb'; P2='#60a5fa'; P3='#1e40af'
TXT='#1e3a5f'; SEC='#6b8aaa'; RED='#dc2626'
YEL='#d97706'; TEA='#0891b2'; EXP='#7c3aed'; DEAD='#1f2937'

plt.rcParams.update({
    'figure.facecolor':BG,   'axes.facecolor':CARD,  'axes.edgecolor':BORD,
    'text.color':TXT,        'axes.labelcolor':SEC,  'xtick.color':SEC,
    'ytick.color':SEC,       'grid.color':'#dde9f7', 'grid.linewidth':0.6,
    'legend.facecolor':CARD, 'legend.edgecolor':BORD,'font.family':'sans-serif',
})

AREA = 50

# ── Realistic outbreak presets (R0, incubation, CFR from published epi data) ─
# Sources: WHO, CDC, published reviews. Params translated to per-step game values.
OUTBREAKS = {
    'covid19': {
        'name': 'COVID-19 (2020, Ancestral)',
        'emoji': '🦠',
        'r0': 2.8, 'incubation_days': 5, 'cfr': 0.015,
        'transmission': 'Respiratory droplets / aerosol',
        'vaccine': True, 'vaccine_delay': 0,
        'inf_radius': 3.6, 'inf_chance': 0.45, 'inc_steps': 5,
        'initial_infected': 2, 'pop_size': 40,
        'context': 'SARS-CoV-2. Moderate R0, long asymptomatic shedding, ~1% CFR. Vaccines arrived ~11 months post-emergence.',
    },
    'ebola': {
        'name': 'Ebola (2014, West Africa)',
        'emoji': '🩸',
        'r0': 1.8, 'incubation_days': 8, 'cfr': 0.50,
        'transmission': 'Direct contact with body fluids',
        'vaccine': True, 'vaccine_delay': 2,
        'inf_radius': 1.8, 'inf_chance': 0.55, 'inc_steps': 8,
        'initial_infected': 1, 'pop_size': 35,
        'context': 'EBOV Makona variant. Low R0 but 50% CFR limits spread — high lethality burns through contact networks fast.',
    },
    'flu_1918': {
        'name': 'Spanish Flu (1918, H1N1)',
        'emoji': '🌬️',
        'r0': 2.5, 'incubation_days': 2, 'cfr': 0.025,
        'transmission': 'Airborne respiratory',
        'vaccine': False, 'vaccine_delay': 0,
        'inf_radius': 3.8, 'inf_chance': 0.50, 'inc_steps': 2,
        'initial_infected': 2, 'pop_size': 45,
        'context': '1918 H1N1 pandemic. Short incubation, rapid spread, no vaccines existed — killed 50M+ globally.',
    },
    'nipah': {
        'name': 'Nipah Virus (Bangladesh)',
        'emoji': '🦇',
        'r0': 0.48, 'incubation_days': 10, 'cfr': 0.72,
        'transmission': 'Bat-to-human, limited H2H',
        'vaccine': False, 'vaccine_delay': 0,
        'inf_radius': 1.4, 'inf_chance': 0.30, 'inc_steps': 10,
        'initial_infected': 1, 'pop_size': 30,
        'context': 'NiV Bangladesh strain. Low R0 but 70%+ CFR. WHO priority pathogen — no licensed vaccine.',
    },
    'measles': {
        'name': 'Measles (Unvaccinated Pop.)',
        'emoji': '👶',
        'r0': 15.0, 'incubation_days': 11, 'cfr': 0.002,
        'transmission': 'Airborne (stays 2h in air)',
        'vaccine': True, 'vaccine_delay': 0,
        'inf_radius': 5.2, 'inf_chance': 0.85, 'inc_steps': 11,
        'initial_infected': 1, 'pop_size': 50,
        'context': 'MeV. The most contagious human virus (R0 12-18). Vaccine is 97% effective — outbreaks today mean coverage gaps.',
    },
    'sars': {
        'name': 'SARS (2003)',
        'emoji': '💨',
        'r0': 2.9, 'incubation_days': 5, 'cfr': 0.096,
        'transmission': 'Respiratory droplets',
        'vaccine': False, 'vaccine_delay': 0,
        'inf_radius': 3.0, 'inf_chance': 0.48, 'inc_steps': 5,
        'initial_infected': 1, 'pop_size': 35,
        'context': 'SARS-CoV-1. ~10% CFR, contained in 8 months by quarantine — no vaccine ever licensed before extinction.',
    },
    'h5n1': {
        'name': 'H5N1 Avian (Pandemic Adapted)',
        'emoji': '🐦',
        'r0': 2.0, 'incubation_days': 3, 'cfr': 0.53,
        'transmission': 'Airborne (hypothetical adaptation)',
        'vaccine': True, 'vaccine_delay': 3,
        'inf_radius': 3.2, 'inf_chance': 0.52, 'inc_steps': 3,
        'initial_infected': 2, 'pop_size': 40,
        'context': 'Hypothetical: if H5N1 gained efficient H2H transmission. Observed CFR in sporadic cases is ~50%. Vaccine exists but scaling takes months.',
    },
}

# ── Agent-based simulation ────────────────────────────────────────────────────
class Agent:
    __slots__ = ['id','x','y','age','lifetime','state','timer','safe','immune']
    # state: 'S' susceptible, 'E' exposed/incubating, 'I' infectious, 'R' recovered, 'D' dead
    def __init__(self, id, x, y, lifetime=80):
        self.id = id; self.x = x; self.y = y
        self.age = 0; self.lifetime = lifetime
        self.state = 'S'; self.timer = 0
        self.safe = False; self.immune = False

class SafeZone:
    def __init__(self, x, y, radius=5, strength=6, capacity=10):
        self.x=x; self.y=y; self.radius=radius; self.strength=strength; self.capacity=capacity

def _d(a, b): return math.hypot(a.x-b.x, a.y-b.y)

def init_pop(size=40, initial_infected=2):
    pop = [Agent(i, random.uniform(0,AREA), random.uniform(0,AREA)) for i in range(size)]
    for a in random.sample(pop, min(initial_infected, len(pop))):
        a.state = 'I'; a.timer = random.randint(6, 12)
    return pop

def move_agents(pop, zones):
    """All agents random-walk (realistic mixing). Zones are opportunistic shelters:
    susceptibles passing nearby enter when capacity allows, infected avoid them."""
    occ = {id(z): 0 for z in zones}
    for a in pop:
        if a.state in ('D','R'): continue
        spd = {'S':1.6, 'E':1.0, 'I':1.4}[a.state]
        # Brownian motion with slight persistence
        dx, dy = random.uniform(-1,1), random.uniform(-1,1)
        m = math.hypot(dx, dy) + 1e-9
        a.x = max(0, min(AREA, a.x + dx/m*spd))
        a.y = max(0, min(AREA, a.y + dy/m*spd))
        # Opportunistic zone entry for non-infectious
        a.safe = False
        if a.state in ('S','E') and zones:
            for z in zones:
                if math.hypot(a.x-z.x, a.y-z.y) <= z.radius and occ[id(z)] < z.capacity:
                    a.safe = True; occ[id(z)] += 1; break

def infect_agents(pop, inf_radius, inf_chance, inc_steps):
    new_exposed = 0
    for a in pop:
        if a.state != 'I': continue
        for o in pop:
            if o.state == 'S' and not o.immune and not o.safe and _d(a, o) <= inf_radius:
                if random.random() < inf_chance:
                    o.state = 'E'; o.timer = inc_steps + random.randint(-1, 2)
                    new_exposed += 1
    return new_exposed

def progress_states(pop, cfr, recovery_steps=10):
    deaths = 0
    for a in pop:
        if a.state == 'E':
            a.timer -= 1
            if a.timer <= 0:
                a.state = 'I'; a.timer = recovery_steps + random.randint(-2, 3)
                a.safe = False
        elif a.state == 'I':
            a.timer -= 1
            # Per-step mortality derived from CFR over infectious duration
            per_step_death = 1 - (1 - cfr) ** (1/recovery_steps)
            if random.random() < per_step_death:
                a.state = 'D'; deaths += 1
            elif a.timer <= 0:
                a.state = 'R'; a.immune = True
    return deaths

def attack_zones(pop, zones):
    breached = 0; remove = []
    for za in [a for a in pop if a.state == 'I']:
        for z in zones:
            if math.hypot(za.x-z.x, za.y-z.y) <= z.radius + 1 and random.random() < 0.12:
                z.strength -= 1
        if z.strength <= 0 and z not in remove: remove.append(z)
    # Exposed inside zones erode them from within when they transition
    for z in zones:
        for a in pop:
            if a.state == 'E' and a.safe and a.timer <= 1 and math.hypot(a.x-z.x, a.y-z.y) <= z.radius + 1:
                z.strength -= 2
        if z.strength <= 0 and z not in remove: remove.append(z)
    for z in remove:
        if z in zones:
            zones.remove(z); breached += 1
            for a in pop: a.safe = False
    return breached

COOLDOWNS = {'vaccinate':5, 'quarantine':3, 'supply':4, 'research':0}

class Game:
    def __init__(self):
        self.pathogen_key = 'covid19'
        self.reset()

    def reset(self, pathogen_key=None):
        if pathogen_key is None: pathogen_key = self.pathogen_key
        self.pathogen_key = pathogen_key
        p = OUTBREAKS[pathogen_key]
        self.pathogen = p
        self.step = 0
        self.game_over = False
        self.outcome = ''
        self.deaths = 0
        self.total_infected = p['initial_infected']
        self.inf_radius = p['inf_radius']
        self.inf_chance = p['inf_chance']
        self.inc_steps = p['inc_steps']
        self.cfr = p['cfr']
        self.population = init_pop(size=p['pop_size'], initial_infected=p['initial_infected'])
        # Zone count scales with R0
        nz = 1 if p['r0'] < 1.0 else (2 if p['r0'] < 3.5 else 3)
        zc = {
            1: [SafeZone(25, 25, radius=6, strength=8, capacity=12)],
            2: [SafeZone(14, 14, radius=4, strength=6, capacity=9), SafeZone(36, 36, radius=4, strength=6, capacity=9)],
            3: [SafeZone(14, 14, radius=3, strength=5, capacity=7), SafeZone(36, 14, radius=3, strength=5, capacity=7), SafeZone(25, 36, radius=3, strength=5, capacity=7)],
        }
        self.zones = zc[nz]
        self.hist = {'t':[], 's':[], 'e':[], 'i':[], 'r':[], 'd':[], 'r0':[]}
        self.cooldowns = {k:0 for k in COOLDOWNS}
        # Vaccines only available if pathogen has them AND not delayed
        self.vaccine_doses = 0 if not p['vaccine'] else max(2, 6 - int(p['r0']))
        self.vaccine_delay_remaining = p['vaccine_delay']
        self.research_progress = 0
        self.msg = f"{p['emoji']} {p['name']} detected. Contain it before it spreads!"
        self.msg_type = 'info'
        self._record()

    def _counts(self):
        s = sum(1 for a in self.population if a.state == 'S')
        e = sum(1 for a in self.population if a.state == 'E')
        i = sum(1 for a in self.population if a.state == 'I')
        r = sum(1 for a in self.population if a.state == 'R')
        return s, e, i, r

    def _record(self):
        s, e, i, r = self._counts()
        self.hist['t'].append(self.step)
        self.hist['s'].append(s); self.hist['e'].append(e)
        self.hist['i'].append(i); self.hist['r'].append(r); self.hist['d'].append(self.deaths)
        r0_est = round(self.hist['i'][-1] / self.hist['i'][-4], 2) if len(self.hist['i']) >= 4 and self.hist['i'][-4] > 0 else 0.0
        self.hist['r0'].append(r0_est)

    def advance(self, action='none'):
        if self.game_over: return
        if action in self.cooldowns and self.cooldowns[action] > 0:
            self.msg = f"Cooldown: {action.capitalize()} ready in {self.cooldowns[action]} step(s)."
            self.msg_type = 'neutral'
            self._tick(); return

        old_chance = self.inf_chance
        applied_quarantine = False
        self.msg_type = 'info'

        if action == 'vaccinate':
            if not self.pathogen['vaccine']:
                self.msg = f"⚠️ No vaccine exists for {self.pathogen['name']}. Use Research to develop one."
                self.msg_type = 'warning'; self._tick(); return
            if self.vaccine_delay_remaining > 0:
                self.msg = f"⏱️ Vaccine rollout in progress: {self.vaccine_delay_remaining} step(s) until shipment."
                self.msg_type = 'warning'; self._tick(); return
            if self.vaccine_doses <= 0:
                self.msg = "No vaccine doses left — use Supply to restock."
                self.msg_type = 'warning'; self._tick(); return
            # Vaccinate susceptibles in zones first, then random
            candidates = [a for a in self.population if a.state == 'S' and not a.immune]
            candidates.sort(key=lambda a: (not a.safe, random.random()))
            dose_count = min(self.vaccine_doses, 5, len(candidates))
            for a in candidates[:dose_count]: a.immune = True
            self.vaccine_doses -= 1
            self.cooldowns['vaccinate'] = COOLDOWNS['vaccinate']
            self.msg = f"💉 Vaccinated {dose_count} people. {self.vaccine_doses} dose(s) left."
            self.msg_type = 'success'
        elif action == 'quarantine':
            self.inf_chance = max(0.04, self.inf_chance * 0.22)
            applied_quarantine = True
            self.cooldowns['quarantine'] = COOLDOWNS['quarantine']
            self.msg = "🚧 Hard quarantine: transmission cut ~78% this step."
            self.msg_type = 'warning'
        elif action == 'supply':
            # Reduce CFR for this step via better care
            saved = 0
            for a in self.population:
                if a.state == 'I' and random.random() < 0.35:
                    a.state = 'R'; a.immune = True; saved += 1
            if self.pathogen['vaccine']:
                self.vaccine_doses = min(self.vaccine_doses + 2, max(4, 8 - int(self.pathogen['r0'])))
            self.cooldowns['supply'] = COOLDOWNS['supply']
            self.msg = f"🏥 Medical supplies: {saved} patient(s) recovered, vaccine stock topped up."
            self.msg_type = 'info'
        elif action == 'research':
            gain = random.randint(12, 22)
            self.research_progress = min(100, self.research_progress + gain)
            self.msg = f"🔬 Research +{gain}% → {self.research_progress}% complete."
            self.msg_type = 'info'
            if self.research_progress >= 100 and not self.pathogen['vaccine']:
                self.pathogen['vaccine'] = True
                self.vaccine_delay_remaining = 0
                self.vaccine_doses = 4
                self.msg = "🧪 Vaccine developed! 4 doses ready to deploy."
                self.msg_type = 'success'
        elif action == 'none':
            self.msg = "⏩ No intervention — pathogen spreads unchecked."
            self.msg_type = 'neutral'

        # Simulation tick
        move_agents(self.population, self.zones)
        new_exposed = infect_agents(self.population, self.inf_radius, self.inf_chance, self.inc_steps)
        self.total_infected += new_exposed
        new_deaths = progress_states(self.population, self.cfr)
        self.deaths += new_deaths
        breached = attack_zones(self.population, self.zones)

        if applied_quarantine: self.inf_chance = old_chance

        self.step += 1
        if self.vaccine_delay_remaining > 0: self.vaccine_delay_remaining -= 1
        self._record(); self._tick()

        # Mutation event
        if self.step > 0 and self.step % random.randint(10, 15) == 0 and random.random() < 0.40:
            self.inf_radius = min(self.inf_radius * 1.22, AREA * 0.4)
            self.inf_chance = min(self.inf_chance * 1.18, 0.95)
            self.msg += "  ⚡ MUTATION: pathogen is more transmissible!"
            self.msg_type = 'danger'

        if breached:
            self.msg += f"  💥 {breached} zone(s) breached!"

        # End conditions
        s, e, i, r = self._counts()
        alive = s + e + i + r
        if alive == 0:
            self.game_over = True; self.outcome = 'failure'
            self.msg = f"☠️ Total population loss. {self.deaths} deaths."
        elif i == 0 and e == 0:
            self.game_over = True; self.outcome = 'success'
            cfr_actual = self.deaths / max(self.total_infected, 1) * 100
            self.msg = f"✅ Pathogen eradicated! {self.deaths} deaths, observed CFR {cfr_actual:.1f}%."
        elif self.step >= 100:
            self.game_over = True; self.outcome = 'partial'
            self.msg = f"⏰ 100 steps elapsed. Survivors: {s+r}/{len(self.population)+self.deaths}, deaths: {self.deaths}."

    def _tick(self):
        for k in self.cooldowns:
            if self.cooldowns[k] > 0: self.cooldowns[k] -= 1

game = Game()


In [3]:
SEC_STYLE = 'style="font-size:0.7rem;font-weight:600;text-transform:uppercase;letter-spacing:0.8px;color:#991b1b;margin:1.2rem 0 0.5rem 0;font-family:Inter,sans-serif;border-bottom:1px solid #c5d6ec;padding-bottom:6px;display:block;"'

header = widgets.HTML(value=(
    '<div class="game-hero">'
    '<div class="game-badge">Outbreak Response Simulator</div>'
    '<h1><span class="accent">Contain</span> the Outbreak</h1>'
    '<p>Pick a real-world pathogen and try to contain it. Each disease has its own R\u2080, incubation period, '
    'case fatality rate, and transmission mode — drawn from published epidemiological data. '
    'Your interventions have real constraints: cooldowns, limited doses, and some diseases have no vaccine yet.</p>'
    '<div class="pill-row">'
    '<span class="game-pill">7 realistic pathogens</span>'
    '<span class="game-pill">SEIRD model</span>'
    '<span class="game-pill">Stochastic mutations</span>'
    '<span class="game-pill">Resource constraints</span>'
    '</div></div>'
))

# ── Outbreak picker ──────────────────────────────────────────────────────────
outbreak_options = [(f"{v['emoji']}  {v['name']}", k) for k, v in OUTBREAKS.items()]
dd_outbreak = widgets.Dropdown(
    options=outbreak_options, value='covid19', description='Pathogen:',
    style={'description_width':'80px'}, layout=widgets.Layout(width='380px'),
)

disease_info_html = widgets.HTML()

def render_disease_card():
    p = OUTBREAKS[dd_outbreak.value]
    vacc_text = 'Available' if p['vaccine'] else 'None (must research)'
    vacc_color = '#059669' if p['vaccine'] else '#dc2626'
    delay_text = f" (+{p['vaccine_delay']} step delay)" if p['vaccine_delay'] > 0 else ''
    disease_info_html.value = (
        f'<div class="disease-card">'
        f'<div class="disease-name">{p["emoji"]} {p["name"]}</div>'
        f'<div class="disease-meta">Transmission: {p["transmission"]}</div>'
        f'<div class="disease-stats">'
        f'<span class="stat-chip">R\u2080 <strong>{p["r0"]}</strong></span>'
        f'<span class="stat-chip">Incubation <strong>{p["incubation_days"]}d</strong></span>'
        f'<span class="stat-chip">CFR <strong>{p["cfr"]*100:.1f}%</strong></span>'
        f'<span class="stat-chip">Initial cases <strong>{p["initial_infected"]}</strong></span>'
        f'<span class="stat-chip">Population <strong>{p["pop_size"]}</strong></span>'
        f'<span class="stat-chip" style="color:{vacc_color}">Vaccine <strong>{vacc_text}{delay_text}</strong></span>'
        f'</div>'
        f'<div class="disease-desc">{p["context"]}</div>'
        f'</div>'
    )

render_disease_card()
dd_outbreak.observe(lambda ch: render_disease_card() if ch['name']=='value' else None, names='value')

# ── Action buttons ───────────────────────────────────────────────────────────
bst = {'font_weight':'600','font_size':'13px'}
bly = widgets.Layout(width='auto', height='36px', margin='0 6px 0 0', padding='0 16px')

btn_vaccine    = widgets.Button(description='Vaccinate',  button_style='success', style=bst, layout=bly)
btn_quarantine = widgets.Button(description='Quarantine', button_style='warning', style=bst, layout=bly)
btn_supply     = widgets.Button(description='Supply',     button_style='info',    style=bst, layout=bly)
btn_research   = widgets.Button(description='Research',   button_style='primary', style=bst, layout=bly)
btn_wait       = widgets.Button(description='Wait',       button_style='',        style=bst, layout=bly)
btn_start      = widgets.Button(description='▶ Start / Reset', button_style='danger', style=bst,
                                 layout=widgets.Layout(width='auto', height='36px', padding='0 20px'))

kpi_html = widgets.HTML()
msg_html = widgets.HTML()
out_plots = widgets.Output(layout=widgets.Layout(width='100%'))

def update_buttons():
    p = game.pathogen
    # Vaccine button
    cd_v = game.cooldowns['vaccinate']
    if not p['vaccine']:
        btn_vaccine.description = 'Vaccinate (none)'; btn_vaccine.disabled = True
    elif game.vaccine_delay_remaining > 0:
        btn_vaccine.description = f'Vaccine in {game.vaccine_delay_remaining}'; btn_vaccine.disabled = True
    elif cd_v > 0:
        btn_vaccine.description = f'💉 Vaccinate ({cd_v})'; btn_vaccine.disabled = True
    else:
        btn_vaccine.description = f'💉 Vaccinate ({game.vaccine_doses})'; btn_vaccine.disabled = (game.vaccine_doses <= 0)
    # Quarantine
    cd_q = game.cooldowns['quarantine']
    btn_quarantine.description = f'🚧 Quarantine ({cd_q})' if cd_q > 0 else '🚧 Quarantine'
    btn_quarantine.disabled = cd_q > 0
    # Supply
    cd_s = game.cooldowns['supply']
    btn_supply.description = f'🏥 Supply ({cd_s})' if cd_s > 0 else '🏥 Supply'
    btn_supply.disabled = cd_s > 0
    # Research — only visible/useful if no vaccine
    if p['vaccine']:
        btn_research.description = 'Research ✓ done'
        btn_research.disabled = True
    else:
        btn_research.description = f'🔬 Research ({game.research_progress}%)'
        btn_research.disabled = False

def render_game():
    s, e, i, r = game._counts()
    tot = s + e + i + r + game.deaths
    p = game.pathogen
    r0_est = game.hist['r0'][-1]
    r0c = '#059669' if r0_est < 1 else ('#d97706' if r0_est < 2 else '#dc2626')
    cfr_obs = (game.deaths / max(game.total_infected, 1)) * 100
    update_buttons()
    kpi_html.value = (
        '<div class="kpi-row">'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Pathogen</div>'
        f'<div class="kpi-bio-val" style="color:#991b1b;font-size:1.1rem">{p["emoji"]} {p["name"].split(" (")[0]}</div>'
        f'<div class="kpi-bio-sub">Step {game.step} / 100</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">R\u2080 (observed)</div>'
        f'<div class="kpi-bio-val" style="color:{r0c}">{r0_est:.2f}</div>'
        f'<div class="kpi-bio-sub">Target: {p["r0"]}</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Susceptible</div>'
        f'<div class="kpi-bio-val" style="color:#2563eb">{s}</div>'
        f'<div class="kpi-bio-sub">Not yet exposed</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Incubating</div>'
        f'<div class="kpi-bio-val" style="color:#7c3aed">{e}</div>'
        f'<div class="kpi-bio-sub">Hidden threat</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Infectious</div>'
        f'<div class="kpi-bio-val" style="color:#dc2626">{i}</div>'
        f'<div class="kpi-bio-sub">Spreading now</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Recovered</div>'
        f'<div class="kpi-bio-val" style="color:#059669">{r}</div>'
        f'<div class="kpi-bio-sub">Immune</div></div>'
        f'<div class="kpi-bio"><div class="kpi-bio-label">Deaths</div>'
        f'<div class="kpi-bio-val" style="color:#1f2937">{game.deaths}</div>'
        f'<div class="kpi-bio-sub">CFR: {cfr_obs:.1f}%</div></div>'
        '</div>'
    )

    bg = {'success':'#dcfce7','warning':'#fef9c3','info':'#dbeafe','neutral':'#f1f5f9','danger':'#fee2e2'}
    bd = {'success':'#16a34a','warning':'#ca8a04','info':'#2563eb','neutral':'#94a3b8','danger':'#dc2626'}
    tc = {'success':'#166534','warning':'#713f12','info':'#1e3a5f','neutral':'#475569','danger':'#991b1b'}
    mt = game.msg_type if game.msg_type in bg else 'info'
    if game.outcome == 'failure': mt = 'danger'
    elif game.outcome == 'success': mt = 'success'
    msg_html.value = (
        f'<div style="background:{bg[mt]};border-left:3px solid {bd[mt]};padding:10px 16px;'
        f'border-radius:6px;color:{tc[mt]};font-size:0.86rem;margin:6px 0;'
        f'font-family:Inter,sans-serif;line-height:1.5;">{game.msg}</div>'
    )

    with out_plots:
        clear_output(wait=True)
        fig = plt.figure(figsize=(13.5, 5.4), facecolor=BG)
        gs = GridSpec(1, 2, figure=fig, wspace=0.28)

        # Map
        ax1 = fig.add_subplot(gs[0]); ax1.set_facecolor(CARD)
        for z in game.zones:
            ax1.add_patch(plt.Circle((z.x, z.y), z.radius, color=P1, fill=True, alpha=0.08))
            ax1.add_patch(plt.Circle((z.x, z.y), z.radius, color=P1, fill=False, lw=1.5, alpha=0.6, ls='--'))
            cap_used = sum(1 for a in game.population if a.safe and a.state != 'D')
            ax1.text(z.x, z.y, f'{cap_used}/{z.capacity}\n{z.strength}hp', ha='center', va='center',
                     fontsize=7, color=P3, alpha=0.85, fontweight='600')
        susc = [(a.x, a.y) for a in game.population if a.state == 'S' and not a.immune]
        immu = [(a.x, a.y) for a in game.population if a.immune and a.state == 'S']
        expo = [(a.x, a.y) for a in game.population if a.state == 'E']
        infe = [(a.x, a.y) for a in game.population if a.state == 'I']
        reco = [(a.x, a.y) for a in game.population if a.state == 'R']
        n_susc, n_immu = len(susc), len(immu)
        if susc: xs, ys = zip(*susc); ax1.scatter(xs, ys, c=P2, s=22, alpha=0.8, zorder=3)
        if immu: xs, ys = zip(*immu); ax1.scatter(xs, ys, c=TEA, s=28, alpha=0.9, zorder=3, marker='P')
        if expo: xs, ys = zip(*expo); ax1.scatter(xs, ys, c=EXP, s=26, alpha=0.88, zorder=4, marker='D')
        if infe: xs, ys = zip(*infe); ax1.scatter(xs, ys, c=RED, s=40, alpha=0.9, zorder=5, marker='X')
        if reco: xs, ys = zip(*reco); ax1.scatter(xs, ys, c='#059669', s=24, alpha=0.85, zorder=3, marker='s')
        ax1.set_xlim(0, AREA); ax1.set_ylim(0, AREA); ax1.set_aspect('equal')
        ax1.set_title(f'Outbreak Map — Step {game.step}/100', color=TXT, fontsize=11, fontweight='600', pad=10)
        ax1.tick_params(labelsize=7)
        for sp in ax1.spines.values(): sp.set_color(BORD)
        ax1.grid(True, alpha=0.4, color='#dde9f7', lw=0.6)
        ax1.legend(handles=[
            mpatches.Patch(color=P2, label=f'Susceptible ({n_susc})'),
            mpatches.Patch(color=TEA, label=f'Immune ({n_immu})'),
            mpatches.Patch(color=EXP, label=f'Incubating ({e})'),
            mpatches.Patch(color=RED, label=f'Infectious ({i})'),
            mpatches.Patch(color='#059669', label=f'Recovered ({r})'),
        ], loc='lower right', fontsize=7, facecolor=CARD, edgecolor=BORD, framealpha=0.95)

        # Dynamics
        ax2 = fig.add_subplot(gs[1]); ax2.set_facecolor(CARD)
        t = game.hist['t']
        ax2.plot(t, game.hist['s'], color=P1, lw=2.0, label='Susceptible', solid_capstyle='round')
        ax2.plot(t, game.hist['e'], color=EXP, lw=1.8, label='Incubating', ls='--')
        ax2.plot(t, game.hist['i'], color=RED, lw=2.2, label='Infectious', solid_capstyle='round')
        ax2.plot(t, game.hist['r'], color='#059669', lw=1.8, label='Recovered', ls=':')
        ax2.plot(t, game.hist['d'], color=DEAD, lw=1.8, label='Deaths', ls='-.')
        ax2r = ax2.twinx()
        ax2r.plot(t, game.hist['r0'], color=YEL, lw=1.4, ls=':', alpha=0.85, label='R\u2080 est.')
        ax2r.axhline(1.0, color=YEL, lw=0.9, ls='--', alpha=0.4)
        ax2r.set_ylabel('R\u2080', color=YEL, fontsize=9); ax2r.tick_params(colors=YEL, labelsize=7.5)
        for sp in ax2r.spines.values(): sp.set_color(BORD)
        ax2r.set_ylim(0, max(max(game.hist['r0'], default=0) + 0.5, 3))
        ax2.set_title('SEIRD Dynamics', color=TXT, fontsize=11, fontweight='600', pad=10)
        ax2.set_xlabel('Step', fontsize=8.5, color=SEC)
        ax2.set_ylabel('Individuals', fontsize=8.5, color=SEC)
        ax2.tick_params(labelsize=7.5)
        for sp in ax2.spines.values(): sp.set_color(BORD)
        ax2.grid(True, alpha=0.4, color='#dde9f7', lw=0.6)
        l1, lab1 = ax2.get_legend_handles_labels()
        l2, lab2 = ax2r.get_legend_handles_labels()
        ax2.legend(l1+l2, lab1+lab2, fontsize=7.5, facecolor=CARD, edgecolor=BORD, loc='upper right', framealpha=0.95)
        plt.tight_layout(pad=1.5); plt.show()

def on_action(action):
    def handler(_):
        game.advance(action); render_game()
    return handler

btn_vaccine.on_click(on_action('vaccinate'))
btn_quarantine.on_click(on_action('quarantine'))
btn_supply.on_click(on_action('supply'))
btn_research.on_click(on_action('research'))
btn_wait.on_click(on_action('none'))

def on_start(_):
    game.reset(pathogen_key=dd_outbreak.value); render_game()
btn_start.on_click(on_start)

sec = lambda t: widgets.HTML(f'<div {SEC_STYLE}>{t}</div>')

picker_row  = widgets.HBox([dd_outbreak, btn_start], layout=widgets.Layout(gap='12px', align_items='center', margin='6px 0'))
action_row  = widgets.HBox([btn_vaccine, btn_quarantine, btn_supply, btn_research, btn_wait],
                            layout=widgets.Layout(margin='4px 0'))

legend_html = widgets.HTML(value=(
    '<div class="disease-card" style="background:#f8fafc;">'
    '<div style="font-size:0.78rem;color:#4a6a8a;line-height:1.6;">'
    '<strong>Legend:</strong>  '
    '<span style="color:#60a5fa">● Susceptible</span>  '
    '<span style="color:#0891b2">✚ Immune (vaccinated)</span>  '
    '<span style="color:#7c3aed">◆ Incubating</span>  '
    '<span style="color:#dc2626">✖ Infectious</span>  '
    '<span style="color:#059669">■ Recovered</span>  '
    '<span style="color:#1f2937">— Deaths tracked in chart</span>'
    '<br><br><strong>Tips:</strong> High-CFR pathogens (Ebola, Nipah) self-limit — focus on zones. '
    'High-R\u2080 (Measles) need vaccine ASAP. No-vaccine pathogens (1918 flu, SARS, Nipah) require Research — expect to lose many before it lands.'
    '</div></div>'
))

dashboard = widgets.VBox([
    header,
    sec('1. Pick your outbreak'), disease_info_html, picker_row,
    sec('2. Live simulation'), kpi_html, out_plots, msg_html,
    sec('3. Intervene'), action_row,
    legend_html,
], layout=widgets.Layout(padding='10px 14px'))

display(dashboard)
render_game()
